# Stage 1 analysis
Reads only from `runs/` (copy them from the VM with rsync). Every figure comes from run folders, using the definitions in `load.py`.

In [ ]:
import sys; sys.path.insert(0, '..')
import matplotlib.pyplot as plt
import pandas as pd
from analysis import load as L
FIG = __import__('pathlib').Path('figures'); FIG.mkdir(exist_ok=True)

## Run inventory and validity

In [ ]:
runs = L.find_runs(valid_only=False)
pd.DataFrame([{'run': r['dir'].name, 'valid': r['checks'] and r['checks']['valid'],
               'failed': r['checks'] and r['checks']['failed']} for r in runs])

## One run over time
Set `RUN`. `SLO_TTFT` is fixed in `docs/decisions.md`.

In [ ]:
RUN = runs[-1]['dir'].name if runs else None
SLO_TTFT = 1.0  # fixed in docs/decisions.md
run = L.load_run(L.RUNS / RUN)
m, g, req = L.rates(run['metrics']), run['gpu'], run['requests']
pf = L.percent_fast(req, SLO_TTFT)
panels = [('percent fast', pf.t, pf.percent_fast), ('hit rate h', m.t, m.get('hit_rate')),
          ('running N', m.t, m.get('num_requests_running')), ('waiting Q', m.t, m.get('num_requests_waiting')),
          ('KV usage', m.t, m.get('kv_cache_usage_perc')),
          ('client lag s', req.t_planned, req.t_sent - req.t_planned),
          ('SM clock MHz', g.get('t'), g.get('sm_clock_mhz'))]
panels = [p for p in panels if p[2] is not None]
fig, axs = plt.subplots(len(panels), 1, figsize=(10, 1.8 * len(panels)), sharex=True)
for ax, (name, x, y) in zip(axs, panels):
    ax.plot(x, y, lw=1); ax.set_ylabel(name, rotation=0, ha='right')
    ax.axvline(run['cfg']['workload']['warmup_s'], color='grey', ls=':')
axs[-1].set_xlabel('t (s from load start)'); fig.suptitle(RUN); fig.tight_layout()
fig.savefig(FIG / f'{RUN}_timeseries.png', dpi=120)

## Repeatability (run 3)
Compare `first_real` and `repeat` within the pre-registered tolerance in `docs/decisions.md`.

In [ ]:
pd.DataFrame([L.summarize(r) for r in runs if r['checks'] and r['checks']['valid'] and r['cfg']['name'] in ('first_real', 'repeat')])

## Load–latency curves: prefix caching on vs off (runs 4–5)

In [ ]:
curve = pd.DataFrame([L.summarize(r) for r in runs if r['checks'] and r['checks']['valid'] and r['cfg'].get('sweep_parent') in ('sweep_pc_on', 'sweep_pc_off')])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for pc, c in curve.groupby('prefix_caching'):
    c = c.sort_values('rate'); lab = 'prefix caching ' + ('on' if pc else 'off')
    a1.plot(c.rate, c.ttft_p99, 'o-', label=lab); a2.plot(c.rate, c.throughput, 'o-', label=lab)
a1.set(xlabel='λ (req/s)', ylabel='TTFT p99 (s)', yscale='log'); a2.plot(curve.rate, curve.rate, 'k:', lw=.8)
a2.set(xlabel='λ (req/s)', ylabel='completions/s'); a1.legend(); fig.tight_layout()
fig.savefig(FIG / 'load_latency.png', dpi=120); curve.sort_values(['prefix_caching', 'rate'])

### μ_warm and μ_off
Capacity thresholds must be written in `docs/decisions.md` **before** this cell is run.

In [ ]:
MIN_TPUT_FRAC, MAX_Q_SLOPE = 0.95, 0.01  # copy from decisions.md
{('mu_warm' if pc else 'mu_off'): L.capacity(c, MIN_TPUT_FRAC, MAX_Q_SLOPE) for pc, c in curve.groupby('prefix_caching')}